# System Metrics Study

In [15]:
from pathlib import Path
from collections import defaultdict

import pandas as pd
import numpy as np

## Experiment Configuration

In [16]:
# dataset = "aitv2"
# scenario = "santos"

In [17]:
dataset = "darpa2000"
scenario = "s1_inside"

In [18]:
experiment_dir = f"../../experiments/{dataset}/{scenario}/logic_study/deepproblog"

In [19]:
experiment = "system_metrics_study"
plots_dir = Path(f"../../reports/{dataset}/{scenario}/{experiment}")
plots_dir.mkdir(parents=True, exist_ok=True)

## Inference Times

In [20]:
metrics_dir = Path(f"{experiment_dir}/metrics")
file_paths = list(metrics_dir.glob("*.npz"))
    
grouped_times = defaultdict(list)
for file_path in file_paths:
    full_name = file_path.stem
    base_name = full_name.rsplit('_', 2)[0]
    data = np.load(file_path, allow_pickle=True)
    grouped_times[base_name].extend(data["inference_times"].tolist())


inference_stats = {}
for base_name, all_times in grouped_times.items():
    avg_time = np.mean(all_times)
    std_time = np.std(all_times)
    
    print(f"{base_name}:")
    print(f"  Runs pooled: {len(all_times)} total inference steps")
    print(f"  Average Time: {avg_time:.6f} seconds")
    print(f"  Std Dev Time: {std_time:.6f} seconds\n")
    
    inference_stats[base_name] = {
        "avg_inference_time": avg_time,
        "std_inference_time": std_time
    }


darpa_temp_context_pretrained_base_w10_full_0.001lr:
  Runs pooled: 113235 total inference steps
  Average Time: 0.005603 seconds
  Std Dev Time: 0.038826 seconds

darpa_temp_baseline_endtoend_base_w10_full_0.001lr:
  Runs pooled: 113235 total inference steps
  Average Time: 0.002190 seconds
  Std Dev Time: 0.021996 seconds

darpa_temp_context_endtoend_base_w10_full_0.001lr:
  Runs pooled: 113235 total inference steps
  Average Time: 0.005287 seconds
  Std Dev Time: 0.040865 seconds

darpa_temp_context_baseline_endtoend_base_w10_full_0.001lr:
  Runs pooled: 113235 total inference steps
  Average Time: 0.003061 seconds
  Std Dev Time: 0.031494 seconds

darpa_temp_endtoend_base_w10_full_0.001lr:
  Runs pooled: 113235 total inference steps
  Average Time: 0.005335 seconds
  Std Dev Time: 0.034691 seconds

darpa_temp_pretrained_base_w10_full_0.001lr:
  Runs pooled: 113235 total inference steps
  Average Time: 0.005278 seconds
  Std Dev Time: 0.036532 seconds



In [21]:
inference_stats_df = pd.DataFrame.from_dict(inference_stats, orient='index')
inference_stats_df.to_csv(plots_dir / "inference_time_stats.csv")

In [22]:
inference_stats_df

,avg_inference_time,std_inference_time
darpa_temp_context_pretrained_base_w10_full_0.001lr,0.005603,0.038826
darpa_temp_baseline_endtoend_base_w10_full_0.001lr,0.002190,0.021996
darpa_temp_context_endtoend_base_w10_full_0.001lr,0.005287,0.040865
darpa_temp_context_baseline_endtoend_base_w10_full_0.001lr,0.003061,0.031494
darpa_temp_endtoend_base_w10_full_0.001lr,0.005335,0.034691
darpa_temp_pretrained_base_w10_full_0.001lr,0.005278,0.036532


## Memory Usage

In [23]:
def load_monitor_csv(file_path):
    try:
        df = pd.read_csv(file_path, parse_dates=['ts'], date_format='%Y-%m-%d %H:%M:%S')
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        df = pd.DataFrame()
    
    if 'ts' in df.columns:
        df['ts'] = pd.to_datetime(df['ts'], format='%Y-%m-%dT%H:%M:%S.%f+00:00Z')
        df['t_s'] = (df['ts'] - df['ts'].iloc[0]).dt.total_seconds()
    else:
        print(f"'ts' column not found in {file_path}.")
    
    return df

In [24]:
def summarize_memory(df, col='rss_mb'):
    secs = df['t_s'].values
    vals = df[col].values
    peak = np.nanmax(vals)
    peak_idx = np.nanargmax(vals)
    time_to_peak = secs[peak_idx]
    auc = np.trapezoid(vals, secs)  # MB·s
    median = np.nanmedian(vals)
    p90 = np.nanpercentile(vals, 90)
    p99 = np.nanpercentile(vals, 99)
    # linear slope MB/s (robust: use first N seconds to estimate steady growth)
    if len(secs) > 5:
        coeff = np.polyfit(secs, vals, 1)[0]
    else:
        coeff = np.nan
    return {
        'peak_mb': float(peak),
        'time_to_peak_s': float(time_to_peak),
        'auc_mb_s': float(auc),
        'median_mb': float(median),
        'p90_mb': float(p90),
        'p99_mb': float(p99),
        'slope_mb_per_s': float(coeff),
    }

In [26]:
# Group records by (model_name, phase) to separate train and test runs
monitor_dir = Path(f"{experiment_dir}/monitor")
file_paths = list(monitor_dir.glob("*.csv"))

grouped_records = defaultdict(list)

for file_path in file_paths:
    experiment_name = file_path.stem
    parts = experiment_name.split("_")
    phase = parts[-2]  # "train" or "test"
    model_name = "_".join(parts[1:-8])
    print(f"Loading {model_name} ({phase})...")
    
    try:
        df = load_monitor_csv(file_path)
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        continue

    summary = summarize_memory(df)
    grouped_records[(model_name, phase)].append(summary)

# Compute averages and std across runs
records = []
for (model_name, phase), summaries in grouped_records.items():
    
    aggregated_metrics = {
        "Model": model_name,
        "Phase": phase,
        "runs": len(summaries)  # Keep track of how many runs were averaged
    }
    
    # Calculate mean and std for every metric returned by summarize_memory()
    for key in summaries[0].keys():
        values = [s[key] for s in summaries]
        aggregated_metrics[f"{key}_mean"] = np.mean(values)
        aggregated_metrics[f"{key}_std"] = np.std(values)
        
    records.append(aggregated_metrics)

# Build DataFrame (no longer using from_dict with orient="index" since we made a list)
results_df = pd.DataFrame(records)

# Optional: format for readability based on the new "_mean" keys
def format_duration(seconds):
    if seconds >= 3600:
        return f"{seconds/3600:.1f}h"
    elif seconds >= 60:
        return f"{seconds/60:.1f}m"
    return f"{seconds:.0f}s"

# Note the updated keys targeting the "_mean" columns
results_df["time_to_peak"] = results_df["time_to_peak_s_mean"].apply(format_duration)
results_df["peak_gb"] = (results_df["peak_mb_mean"] / 1024).round(2)
results_df["peak_gb_std"] = (results_df["peak_mb_std"] / 1024).round(2)
results_df["median_gb"] = (results_df["median_mb_mean"] / 1024).round(2)

# Pivot or display side-by-side
display_cols = [
    "Model", "Phase", "runs", 
    "peak_gb", "peak_gb_std", 
    "median_gb", "time_to_peak", 
    "slope_mb_per_s_mean"
]
print(results_df[display_cols])


Loading temp_pretrained (test)...
Loading temp_baseline_endtoend (test)...
Loading temp_endtoend (train)...
Loading temp_pretrained (train)...
Loading temp_context_endtoend (test)...
Loading temp_context_pretrained (test)...
Loading temp_context_baseline_endtoend (train)...
Loading temp_context_baseline_endtoend (test)...
Loading temp_endtoend (train)...
Loading temp_pretrained (test)...
Loading temp_baseline_endtoend (train)...
Loading temp_context_pretrained (test)...
Loading temp_context_endtoend (train)...
Loading temp_context_endtoend (train)...
Loading temp_baseline_endtoend (train)...
Loading temp_baseline_endtoend (test)...
Loading temp_pretrained (train)...
Error loading ../../experiments/darpa2000/s1_inside/logic_study/deepproblog/monitor/darpa_temp_pretrained_base_w10_full_0.001lr_20260614_050149_train_metrics.csv: time data "2026-06-14T03:24:26+00:00Z" doesn't match format "%Y-%m-%dT%H:%M:%S.%f+00:00Z", at position 1332. You might want to try:
    - passing `format` if your

In [27]:
results_df.to_csv(f"{plots_dir}/system_metrics_summary.csv", index=False)

In [28]:
results_df[results_df["Phase"] == "test"]

,Model,Phase,runs,peak_mb_mean,peak_mb_std,time_to_peak_s_mean,time_to_peak_s_std,auc_mb_s_mean,auc_mb_s_std,median_mb_mean,...,p90_mb_mean,p90_mb_std,p99_mb_mean,p99_mb_std,slope_mb_per_s_mean,slope_mb_per_s_std,time_to_peak,peak_gb,peak_gb_std,median_gb
0,temp_pretrained,test,3,15692.048177,21.429739,202.631580,33.712712,2.292894e+06,385453.613155,11049.826172,...,14914.276823,141.155437,15680.950156,19.152739,44.573384,7.869395,3.4m,15.32,0.02,10.79
1,temp_baseline_endtoend,test,3,13302.679688,13.273909,84.221227,19.248216,8.487355e+05,197005.293442,9828.109375,...,12634.107943,33.177109,13252.205299,30.958549,75.750970,19.826709,1.4m,12.99,0.01,9.60
4,temp_context_endtoend,test,3,15566.610677,5.561719,202.123051,16.932315,2.290261e+06,162902.171450,10908.359375,...,14696.165365,159.382121,15526.845651,3.208321,39.039605,4.323903,3.4m,15.20,0.01,10.65
5,temp_context_pretrained,test,3,15551.591146,4.353384,213.963766,26.902417,2.444565e+06,333914.508248,11217.026693,...,14694.122526,79.009524,15513.719154,17.338812,37.321992,4.610692,3.6m,15.19,0.00,10.95
7,temp_context_baseline_endtoend,test,4,12955.749023,651.128619,123.646208,20.434959,1.246813e+06,203975.820933,9848.802246,...,12419.008691,433.842624,12911.663760,626.896992,50.247227,10.923469,2.1m,12.65,0.64,9.62
10,temp_endtoend,test,3,15678.677083,3.357814,204.312524,22.667962,2.272335e+06,258288.685666,10574.750000,...,14813.532552,86.829960,15671.096589,5.553038,41.953546,5.557873,3.4m,15.31,0.00,10.33


In [30]:
results_df[results_df["Phase"] == "test"]["peak_gb"]

0     15.32
1     12.99
4     15.20
5     15.19
7     12.65
10    15.31
Name: peak_gb, dtype: float64

In [29]:
results_df[results_df["Phase"] == "test"]["peak_gb_std"]

0     0.02
1     0.01
4     0.01
5     0.00
7     0.64
10    0.00
Name: peak_gb_std, dtype: float64